# MAAI901 - Applied Vision Transformer Report
## Fine-tuning a Vision Transformer (ViT) on MNIST, compared with the k-NN and MLP baselines

**How to run:** Google Colab -> Runtime -> Change runtime type -> **T4 GPU** -> Run all.
Set `QUICK = True` in the config cell first if you only want a fast test run (about 5 min).

**Pipeline (same protocol as the dashboard project):**
official 60k/10k split -> stratified 54k/6k validation split (seed 42) -> choose settings on validation only -> test set used **once** at the end.


In [ ]:
!pip -q install transformers datasets accelerate scikit-learn matplotlib

In [ ]:
import os, time, json, random
import numpy as np, torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from torchvision import transforms as T
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from transformers import (AutoImageProcessor, ViTForImageClassification,
                          TrainingArguments, Trainer)

# ---------------- CONFIG ----------------
QUICK      = False          # True = small subset, fast smoke test
SEED       = 42
MODEL_NAME = "google/vit-base-patch16-224-in21k"
EPOCHS     = 2
BATCH      = 64
LR         = 3e-5
OUT        = "outputs"
os.makedirs(OUT, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Dataset and split
MNIST: 60,000 train / 10,000 test, 28x28 grayscale, 10 classes. The 10% validation split is stratified and fixed with seed 42.
The test set is never used for tuning.

In [ ]:
raw = load_dataset("ylecun/mnist")
split = raw["train"].train_test_split(test_size=6000, stratify_by_column="label", seed=SEED)
train_ds, val_ds, test_ds = split["train"], split["test"], raw["test"]

if QUICK:
    train_ds = train_ds.shuffle(seed=SEED).select(range(6000))
    val_ds   = val_ds.select(range(1000))
    test_ds  = test_ds.select(range(2000))
    EPOCHS = 1

print(len(train_ds), len(val_ds), len(test_ds))
print("Class counts (train):", np.bincount(train_ds["label"]))

fig, ax = plt.subplots(2, 10, figsize=(15, 3.2))
for d in range(10):
    idx = [i for i, l in enumerate(train_ds["label"][:3000]) if l == d][:2]
    for r, i in enumerate(idx):
        ax[r, d].imshow(train_ds[i]["image"], cmap="gray"); ax[r, d].axis("off")
        if r == 0: ax[r, d].set_title(str(d))
plt.tight_layout(); plt.savefig(f"{OUT}/fig_samples.png", dpi=150); plt.show()

## 2. Pre-processing (tailored to the ViT)
ViT-Base/16 expects **224x224 RGB** input normalised with the model's own mean/std. MNIST is 28x28 grayscale, so each image is:
upscaled to 224x224 -> grayscale replicated to 3 channels -> normalised with the processor's mean/std.
**Augmentation (training only):** small rotation (+-10 degrees), shift (10%), scale (0.9-1.1) to imitate different handwriting styles.
The earlier k-NN/MLP pipeline had no augmentation; this is a new addition.

In [ ]:
proc = AutoImageProcessor.from_pretrained(MODEL_NAME)
mean, std = proc.image_mean, proc.image_std
print("mean/std:", mean, std)

norm_tail = [T.Resize((224, 224)), T.Grayscale(num_output_channels=3), T.ToTensor(), T.Normalize(mean, std)]
train_tf = T.Compose([T.RandomAffine(10, translate=(0.1, 0.1), scale=(0.9, 1.1))] + norm_tail)
eval_tf  = T.Compose(norm_tail)

def make_set_transform(tf):
    def fn(batch):
        batch["pixel_values"] = [tf(img.convert("L")) for img in batch["image"]]
        return batch
    return fn

# keep raw PIL copies for baselines / robustness tests before attaching transforms
def to_np(ds): return np.stack([np.array(im) for im in ds["image"]]).astype(np.float32)
Xtr_np, ytr = to_np(train_ds), np.array(train_ds["label"])
Xva_np, yva = to_np(val_ds),   np.array(val_ds["label"])
Xte_np, yte = to_np(test_ds),  np.array(test_ds["label"])
test_images_pil = list(test_ds["image"])   # for robustness section

train_ds.set_transform(make_set_transform(train_tf))
val_ds.set_transform(make_set_transform(eval_tf))
test_ds.set_transform(make_set_transform(eval_tf))

## 3. Model and configuration
**Why ViT-Base/16 (in21k):** it is pretrained on ImageNet-21k, so it already knows general visual features (edges, strokes, shapes). Fine-tuning transfers this to digits with little data/time.
It splits the image into 16x16 patches and uses self-attention across patches (Dosovitskiy et al., 2021).
A new 10-class head replaces the pretrained head. All layers are fine-tuned.

| Setting | Value | Reason |
|---|---|---|
| Learning rate | 3e-5 | small LR protects pretrained weights |
| Batch size | 64 | fits a T4 with fp16 |
| Epochs | 2 | MNIST is easy; avoids overfitting and saves time |
| Optimiser | AdamW, weight decay 0.01 | standard for transformers |
| Warm-up | 10% of steps | stabilises early training |
| Precision | fp16 | speed on GPU |
| Seed | 42 | reproducible |

In [ ]:
id2label = {i: str(i) for i in range(10)}
model = ViTForImageClassification.from_pretrained(
    MODEL_NAME, num_labels=10, id2label=id2label, label2id={v: k for k, v in id2label.items()})
print("Parameters (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 1))

## 4. Fine-tuning

In [ ]:
def collate(batch):
    return {"pixel_values": torch.stack([b["pixel_values"] for b in batch]),
            "labels": torch.tensor([b["label"] for b in batch])}

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    pr, rc, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(p.label_ids, preds), "precision_macro": pr, "recall_macro": rc, "f1_macro": f1}

import math
WARMUP = int(0.1 * math.ceil(len(train_ds) / BATCH) * EPOCHS)
args = TrainingArguments(
    output_dir="vit_ckpt", per_device_train_batch_size=BATCH, per_device_eval_batch_size=128,
    num_train_epochs=EPOCHS, learning_rate=LR, weight_decay=0.01, warmup_steps=WARMUP,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="accuracy", logging_steps=50, fp16=torch.cuda.is_available(),
    remove_unused_columns=False, report_to="none", seed=SEED, save_total_limit=1)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=collate, compute_metrics=compute_metrics)

t0 = time.time(); trainer.train(); vit_train_seconds = time.time() - t0
print(f"ViT training time: {vit_train_seconds/60:.1f} min")
val_metrics = trainer.evaluate()
print("Validation:", {k: round(v, 4) for k, v in val_metrics.items() if k.startswith("eval_") and isinstance(v, float)})

In [ ]:
log = trainer.state.log_history
tr = [(l["step"], l["loss"]) for l in log if "loss" in l]
ev = [(l["epoch"], l["eval_accuracy"]) for l in log if "eval_accuracy" in l]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(*zip(*tr)); ax[0].set_title("Training loss"); ax[0].set_xlabel("step"); ax[0].set_ylabel("loss")
ax[1].plot(*zip(*ev), marker="o"); ax[1].set_title("Validation accuracy"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_training_curves.png", dpi=150); plt.show()

## 5. Baselines on the same split (k-NN and MLP from the dashboard project)
Settings are the project defaults: k-NN k=3, uniform, Euclidean; MLP 1 hidden layer (64), ReLU, Adam, early stopping.
Pixels are flattened and divided by 255, exactly as in the earlier pipeline.

In [ ]:
Xtr_f, Xva_f, Xte_f = [x.reshape(len(x), -1) / 255.0 for x in (Xtr_np, Xva_np, Xte_np)]

t0 = time.time(); knn = KNeighborsClassifier(n_neighbors=3, weights="uniform", p=2, n_jobs=-1).fit(Xtr_f, ytr)
knn_train_s = time.time() - t0
t0 = time.time(); knn_pred = knn.predict(Xte_f); knn_inf_ms = (time.time() - t0) * 1000 / len(Xte_f)

t0 = time.time()
mlp = MLPClassifier(hidden_layer_sizes=(64,), activation="relu", solver="adam", batch_size=256,
                    learning_rate_init=1e-3, alpha=1e-4, max_iter=30, early_stopping=True,
                    n_iter_no_change=3, random_state=SEED).fit(Xtr_f, ytr)
mlp_train_s = time.time() - t0
t0 = time.time(); mlp_pred = mlp.predict(Xte_f); mlp_inf_ms = (time.time() - t0) * 1000 / len(Xte_f)
print("MLP iterations:", mlp.n_iter_, "(cap 30)")

## 6. Final held-out test evaluation (run once)

In [ ]:
t0 = time.time(); out = trainer.predict(test_ds); vit_inf_ms = (time.time() - t0) * 1000 / len(test_ds)
vit_logits = out.predictions; vit_pred = vit_logits.argmax(-1)

def metrics(y, p):
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average="macro", zero_division=0)
    return dict(accuracy=accuracy_score(y, p), precision_macro=pr, recall_macro=rc, f1_macro=f1)

results = {
    "k-NN (k=3)":      {**metrics(yte, knn_pred), "train_s": knn_train_s, "inference_ms_per_img": knn_inf_ms},
    "MLP (64 hidden)": {**metrics(yte, mlp_pred), "train_s": mlp_train_s, "inference_ms_per_img": mlp_inf_ms},
    "ViT-B/16 fine-tuned": {**metrics(yte, vit_pred), "train_s": vit_train_seconds, "inference_ms_per_img": vit_inf_ms},
}
print(f"{'Model':22s} {'Acc':>7s} {'P':>7s} {'R':>7s} {'F1':>7s} {'train s':>9s} {'ms/img':>8s}")
for k, v in results.items():
    print(f"{k:22s} {v['accuracy']:.4f} {v['precision_macro']:.4f} {v['recall_macro']:.4f} {v['f1_macro']:.4f} {v['train_s']:9.1f} {v['inference_ms_per_img']:8.3f}")
json.dump(results, open(f"{OUT}/results.json", "w"), indent=2)

print("\nViT per-class report:\n", classification_report(yte, vit_pred, digits=4))

In [ ]:
# Confusion matrix (ViT)
cm = confusion_matrix(yte, vit_pred)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm, cmap="Blues"); plt.colorbar(im)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("ViT confusion matrix (test)")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cm[i, j] > cm.max()/2 else "black")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_confusion_vit.png", dpi=150); plt.show()

# Model comparison chart
names = list(results); accs = [results[n]["accuracy"] * 100 for n in names]
plt.figure(figsize=(6, 3.5)); b = plt.bar(names, accs, color=["#999", "#6a9", "#36c"])
plt.ylim(min(accs) - 1, 100); plt.ylabel("Test accuracy (%)"); plt.title("Model comparison")
for r, a in zip(b, accs): plt.text(r.get_x() + r.get_width()/2, a + 0.05, f"{a:.2f}", ha="center")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_model_comparison.png", dpi=150); plt.show()

# Per-class recall (which digits are weakest)
rec = cm.diagonal() / cm.sum(1)
plt.figure(figsize=(6, 3)); plt.bar(range(10), rec * 100); plt.xticks(range(10))
plt.ylim(max(0, rec.min()*100 - 2), 100.2); plt.xlabel("Digit"); plt.ylabel("Recall (%)"); plt.title("ViT per-class recall")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_per_class_recall.png", dpi=150); plt.show()

# Most confused pairs
pairs = [(cm[i, j], i, j) for i in range(10) for j in range(10) if i != j]
print("Top confusions (count, true->pred):", sorted(pairs, reverse=True)[:5])

In [ ]:
# Sample outputs: errors and correct predictions with confidence
probs = torch.softmax(torch.tensor(vit_logits), -1).numpy()
wrong = np.where(vit_pred != yte)[0]; right = np.where(vit_pred == yte)[0]
rng = np.random.default_rng(SEED)
pick_w = rng.choice(wrong, min(12, len(wrong)), replace=False) if len(wrong) else []
pick_r = rng.choice(right, 12, replace=False)
fig, ax = plt.subplots(4, 6, figsize=(12, 8.4))
for a in ax.ravel(): a.axis("off")
for k, i in enumerate(list(pick_r) + list(pick_w)):
    a = ax.ravel()[k]; a.imshow(Xte_np[i], cmap="gray")
    ok = vit_pred[i] == yte[i]
    a.set_title(f"T:{yte[i]} P:{vit_pred[i]} ({probs[i].max():.2f})", color="green" if ok else "red", fontsize=9)
plt.suptitle("Sample ViT predictions (green = correct, red = wrong; T=true, P=predicted, confidence)")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_samples_predictions.png", dpi=150); plt.show()
print("Total test errors:", len(wrong))

## 7. Bias and robustness analysis
MNIST digits are centred, clean and written by a limited group of writers. Real handwriting differs. Here the **test images are modified** (rotation, shift, noise, thicker/thinner strokes) to see how each model copes with a style shift.
A large accuracy drop = the model learned MNIST's style, not digits in general.

In [ ]:
from PIL import ImageFilter
N = min(2000, len(test_images_pil))
sub_imgs, sub_y = test_images_pil[:N], yte[:N]
rs = np.random.default_rng(SEED)

def rot(im):   return T.functional.rotate(im, 20)
def shift(im): return T.functional.affine(im, angle=0, translate=(5, 5), scale=1.0, shear=0)
def noise(im):
    a = np.array(im).astype(np.float32) + rs.normal(0, 40, (28, 28)); return T.functional.to_pil_image(np.clip(a, 0, 255).astype(np.uint8))
def thick(im): return im.filter(ImageFilter.MaxFilter(3))
def thin(im):  return im.filter(ImageFilter.MinFilter(3))
perturb = {"original": lambda im: im, "rotate 20deg": rot, "shift 5px": shift,
           "gaussian noise": noise, "thicker strokes": thick, "thinner strokes": thin}

@torch.no_grad()
def vit_predict(imgs):
    model.eval(); dev = next(model.parameters()).device; preds = []
    for i in range(0, len(imgs), 128):
        x = torch.stack([eval_tf(im) for im in imgs[i:i+128]]).to(dev)
        with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
            preds.append(model(pixel_values=x).logits.argmax(-1).cpu())
    return torch.cat(preds).numpy()

def flat(imgs): return np.stack([np.array(im) for im in imgs]).reshape(len(imgs), -1).astype(np.float32) / 255.0

rob = {m: [] for m in ["k-NN", "MLP", "ViT"]}
for name, fn in perturb.items():
    imgs = [fn(im.convert("L")) for im in sub_imgs]
    rob["k-NN"].append(accuracy_score(sub_y, knn.predict(flat(imgs))) * 100)
    rob["MLP"].append(accuracy_score(sub_y, mlp.predict(flat(imgs))) * 100)
    rob["ViT"].append(accuracy_score(sub_y, vit_predict(imgs)) * 100)
    print(f"{name:16s}", {m: round(v[-1], 1) for m, v in rob.items()})

x = np.arange(len(perturb)); w = 0.27
plt.figure(figsize=(10, 4))
for k, (m, v) in enumerate(rob.items()): plt.bar(x + (k-1)*w, v, w, label=m)
plt.xticks(x, perturb.keys(), rotation=20); plt.ylabel("Accuracy (%)"); plt.legend()
plt.title("Robustness to handwriting-style shifts (first %d test images)" % N)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_robustness.png", dpi=150); plt.show()
json.dump(rob, open(f"{OUT}/robustness.json", "w"), indent=2)

In [ ]:
# Confidence analysis: is the model over-confident when it is wrong?
conf = probs.max(1)
print(f"Mean confidence | correct: {conf[right].mean():.3f}   wrong: {conf[wrong].mean() if len(wrong) else float('nan'):.3f}")
plt.figure(figsize=(6, 3.2))
plt.hist(conf[right], bins=20, alpha=.7, label="correct"); 
if len(wrong): plt.hist(conf[wrong], bins=20, alpha=.7, label="wrong")
plt.yscale("log"); plt.xlabel("Softmax confidence"); plt.legend(); plt.title("Confidence: correct vs wrong")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_confidence.png", dpi=150); plt.show()

trainer.save_model(f"{OUT}/vit_mnist_final")
print("Saved. Download the 'outputs' folder (figures + results.json + model).")

## 8. Notes to use in the report (edit with your own numbers)
- **Dataset:** MNIST - standard benchmark, clean labels, lets a fair comparison with the earlier k-NN/MLP baselines; limitation = easy, centred, limited writer diversity.
- **Preprocessing:** resize 224, 3-channel, ViT mean/std normalisation, training-only affine augmentation.
- **Model:** ViT-B/16 in21k pretrained, full fine-tuning, AdamW 3e-5, 2 epochs, fp16, seed 42.
- **Evaluation:** accuracy, macro P/R/F1, confusion matrix, per-class recall, training/inference time, robustness tests, confidence analysis.
- **Ethics:** writer/cultural variation, no per-writer fairness test, over-confidence, high-stakes use (cheques, forms) needs human review, compute/energy cost of ViT vs k-NN.
- **Honest limits:** MNIST near-saturated, so ViT gain over baselines is small; robustness results matter more than headline accuracy.
